# LeakBench Polyp Segmentation Benchmark

Evaluates 5 segmentation/detection models (EndoFM-LV, EndoViT, ChakraTransformerSegmenter, YOLOv8m, YOLOv8m-seg) on the LeakBench fixed test subsets (ClinicDB, ColonDB, CVC-300, ETIS, Kvasir) plus a false-positive check on `negatives_polypgen_v1`.

**Note on EndoFM-LV / EndoViT loading:** each repo's public checkpoint format and model class name can change between commits. The loader below tries the repo's own API first and prints a clear warning and falls back to a plain ViT encoder + lightweight decode head (untrained) if that import fails, so the notebook always completes and the failure is visible in the output/log rather than silently producing wrong numbers. Before trusting EndoFM-LV/EndoViT scores, check the printed `[model] repo load failed` / `[model] loaded` lines and, if needed, update the import path in `RepoModelWrapper._try_load_repo` to match the cloned repo's actual module layout.


In [ ]:
import os, sys, subprocess

def run(cmd):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=False)

run("pip install -q ultralytics timm albumentations opencv-python-headless scipy pyarrow gdown tabulate")

WORK = "/kaggle/working"
os.makedirs(WORK, exist_ok=True)

REPOS = {
    "EndoFM-LV": "https://github.com/med-air/EndoFM-LV",
    "EndoViT": "https://github.com/DominikBatic/EndoViT",
}

for name, url in REPOS.items():
    dest = os.path.join(WORK, name)
    if not os.path.isdir(dest):
        run(f"git clone --depth 1 {url} {dest}")
    if dest not in sys.path:
        sys.path.insert(0, dest)

print("Repo paths on sys.path:")
for p in sys.path:
    if WORK in p:
        print(" ", p)


In [ ]:
import glob
import torch

LEAKBENCH_ROOT = "/kaggle/input/datasets/gokulrocky/chakra-leakbench-v1/fixed_polyp_dataset_v1"
TEST_ROOT = os.path.join(LEAKBENCH_ROOT, "supervised", "test")
NEG_ROOT = os.path.join(LEAKBENCH_ROOT, "negatives_polypgen_v1")

SUBSETS = ["kvasir", "clinicdb", "colondb", "cvc300", "etis"]

WEIGHTS = {
    "chakra_transformer": "/kaggle/input/datasets/gokulraj324/chakramodel-weightsupdated4/chakra_transformer_best.pth",
    "yolov8m": '/kaggle/input/datasets/gokulraj324/yolo-weight/best (3).pt',
    "yolov8m_seg_dir": "/kaggle/input/datasets/varshasrao/yoloseg/yolov8m-seg",
}

IMG_SIZE = 384
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

for k, v in WEIGHTS.items():
    exists = os.path.exists(v) if not k.endswith("_dir") else os.path.isdir(v)
    print(f"{k}: {v} -> {'found' if exists else 'MISSING'}")


## Model definitions and unified inference engine

In [ ]:
for p in glob.glob("/kaggle/input/**/negatives_polypgen_v1*", recursive=True):
    print(p)

In [ ]:
import os

print("--- Checking LEAKBENCH_ROOT ---")
print("Exists:", os.path.exists(LEAKBENCH_ROOT))
if os.path.exists(LEAKBENCH_ROOT):
    for root, dirs, files in os.walk(LEAKBENCH_ROOT):
        depth = root[len(LEAKBENCH_ROOT):].count(os.sep)
        if depth <= 3:
            sample_files = [f for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg'))][:3]
            print(f"Directory: {root}")
            print(f"  Subdirs: {dirs}")
            print(f"  Sample files: {sample_files}")

In [ ]:
import torch.nn as nn
import timm


class ChakraTransformerSegmenter(nn.Module):
    """ViT-L/16 @384 encoder + 2-stage transposed-conv decode head,
    matching the ChakraTransformer weights spec exactly."""

    def __init__(self, img_size=384, patch_size=16, embed_dim=1024):
        super().__init__()
        self.encoder = timm.create_model(
            "vit_large_patch16_384",
            pretrained=False,
            img_size=img_size,
            num_classes=0,
        )
        self.patch_size = patch_size
        self.grid = img_size // patch_size  # 24 for 384/16

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(embed_dim, 256, kernel_size=4, stride=4),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Dropout2d(0.1),
            nn.ConvTranspose2d(256, 64, kernel_size=4, stride=4),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Dropout2d(0.1),
            nn.Conv2d(64, 1, kernel_size=3, padding=1),
        )

    def forward(self, x):
        feats = self.encoder.forward_features(x)
        if feats.dim() == 3:
            feats = feats[:, 1:, :]  # drop CLS token
            B, N, C = feats.shape
            feats = feats.transpose(1, 2).reshape(B, C, self.grid, self.grid)
        return self.decoder(feats)


class RepoModelWrapper(nn.Module):
    """Generic wrapper for a cloned segmentation repo. Tries the repo's own
    model class + checkpoint first; falls back to a plain ViT-B/16 encoder
    with a lightweight decode head (untrained) if the repo API can't be
    resolved, so inference never crashes the benchmark loop."""

    def __init__(self, name, repo_dir, ckpt_candidates, img_size=384,
                 fallback_arch="vit_base_patch16_384"):
        super().__init__()
        self.name = name
        self.img_size = img_size
        self.model = None
        self.mode = None

        if repo_dir and os.path.isdir(repo_dir):
            self.model = self._try_load_repo(name, repo_dir, ckpt_candidates)
            if self.model is not None:
                self.mode = "repo"

        if self.model is None:
            print(f"[{name}] repo load unavailable -> using ViT fallback encoder/decoder")
            self.model = self._build_fallback(fallback_arch)
            self.mode = "fallback"

    def _try_load_repo(self, name, repo_dir, ckpt_candidates):
        ckpt_path = next((c for c in ckpt_candidates if c and os.path.exists(c)), None)
        try:
            if name == "EndoFM-LV":
                from models.endofm_lv import EndoFMLV  # adjust to match repo module layout
                m = EndoFMLV(pretrained=False)
            elif name == "EndoViT":
                from endovit.model import EndoViTSegmenter  # adjust to match repo module layout
                m = EndoViTSegmenter(img_size=self.img_size)
            else:
                return None
            if ckpt_path:
                state = torch.load(ckpt_path, map_location="cpu")
                if isinstance(state, dict) and "state_dict" in state:
                    state = state["state_dict"]
                m.load_state_dict(state, strict=False)
                print(f"[{name}] loaded checkpoint {ckpt_path}")
            else:
                print(f"[{name}] repo class loaded, no checkpoint found -> using random init")
            return m
        except Exception as e:
            print(f"[{name}] repo load failed ({e}); falling back")
            return None

    def _build_fallback(self, arch):
        encoder = timm.create_model(arch, pretrained=False, img_size=self.img_size, num_classes=0)
        embed_dim = encoder.num_features
        grid = self.img_size // 16
        decoder = nn.Sequential(
            nn.ConvTranspose2d(embed_dim, 128, kernel_size=4, stride=4),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128, 32, kernel_size=4, stride=4),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, kernel_size=3, padding=1),
        )
        return nn.ModuleDict({"encoder": encoder, "decoder": decoder})

    def forward(self, x):
        if self.mode == "repo":
            out = self.model(x)
            if isinstance(out, dict):
                out = out.get("logits", out.get("mask", list(out.values())[0]))
            return out
        feats = self.model["encoder"].forward_features(x)
        if feats.dim() == 3:
            feats = feats[:, 1:, :]
            B, N, C = feats.shape
            g = int(N ** 0.5)
            feats = feats.transpose(1, 2).reshape(B, C, g, g)
        return self.model["decoder"](feats)


In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


def preprocess(img_bgr, size=IMG_SIZE):
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(img_rgb, (size, size), interpolation=cv2.INTER_LINEAR)
    t = torch.from_numpy(resized).float().permute(2, 0, 1) / 255.0
    t = (t - IMAGENET_MEAN) / IMAGENET_STD
    return t.unsqueeze(0)


class ModelInferenceEngine:
    """Loads all 5 models and exposes a single .predict(name, img_bgr) ->
    float32 probability map resized to the original image's H x W."""

    def __init__(self, device=DEVICE):
        self.device = device
        self.models = {}
        self._load_all()

    def _load_all(self):
        self.models["EndoFM-LV"] = RepoModelWrapper(
            "EndoFM-LV", os.path.join("/kaggle/working", "EndoFM-LV"), [None],
        ).to(self.device).eval()

        self.models["EndoViT"] = RepoModelWrapper(
            "EndoViT", os.path.join("/kaggle/working", "EndoViT"), [None],
        ).to(self.device).eval()

        chakra = ChakraTransformerSegmenter(img_size=IMG_SIZE)
        ckpt = WEIGHTS["chakra_transformer"]
        if os.path.exists(ckpt):
            try:
                state = torch.load(ckpt, map_location="cpu")
                if isinstance(state, dict) and "state_dict" in state:
                    state = state["state_dict"]
                chakra.load_state_dict(state, strict=False)
                print("[ChakraTransformerSegmenter] loaded", ckpt)
            except Exception as e:
                print("[ChakraTransformerSegmenter] load failed:", e)
        else:
            print("[ChakraTransformerSegmenter] checkpoint not found, using random init")
        self.models["ChakraTransformerSegmenter"] = chakra.to(self.device).eval()

        if os.path.exists(WEIGHTS["yolov8m"]):
            self.models["YOLOv8m"] = YOLO(WEIGHTS["yolov8m"])
        else:
            print("[YOLOv8m] weights not found:", WEIGHTS["yolov8m"])
            self.models["YOLOv8m"] = None

        seg_dir = WEIGHTS["yolov8m_seg_dir"]
        seg_weight = None
        if os.path.isdir(seg_dir):
            cands = glob.glob(os.path.join(seg_dir, "*.pt"))
            seg_weight = cands[0] if cands else None
        if seg_weight:
            self.models["YOLOv8m-seg"] = YOLO(seg_weight)
        else:
            print("[YOLOv8m-seg] weights not found under", seg_dir)
            self.models["YOLOv8m-seg"] = None

    @torch.no_grad()
    def _infer_torch(self, model, img_bgr):
        h, w = img_bgr.shape[:2]
        x = preprocess(img_bgr).to(self.device)
        logits = model(x)
        prob = torch.sigmoid(logits)
        prob = torch.nn.functional.interpolate(prob, size=(h, w), mode="bilinear", align_corners=False)
        return prob.squeeze().float().cpu().numpy()

    @torch.no_grad()
    def _infer_yolo(self, model, img_bgr):
        h, w = img_bgr.shape[:2]
        prob = np.zeros((h, w), dtype=np.float32)
        if model is None:
            return prob
        results = model.predict(img_bgr, verbose=False, device=self.device)
        r = results[0]
        if getattr(r, "masks", None) is not None:
            for m in r.masks.data.cpu().numpy():
                m_rs = cv2.resize(m, (w, h), interpolation=cv2.INTER_LINEAR)
                prob = np.maximum(prob, m_rs)
        elif getattr(r, "boxes", None) is not None and len(r.boxes) > 0:
            for box, conf in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()):
                x1, y1, x2, y2 = [int(v) for v in box]
                x1, y1 = max(0, x1), max(0, y1)
                x2, y2 = min(w, x2), min(h, y2)
                prob[y1:y2, x1:x2] = np.maximum(prob[y1:y2, x1:x2], float(conf))
        return prob

    def predict(self, model_name, img_bgr):
        model = self.models.get(model_name)
        if model_name in ("YOLOv8m", "YOLOv8m-seg"):
            return self._infer_yolo(model, img_bgr)
        if model is None:
            return np.zeros(img_bgr.shape[:2], dtype=np.float32)
        return self._infer_torch(model, img_bgr)


MODEL_NAMES = ["EndoFM-LV", "EndoViT", "ChakraTransformerSegmenter", "YOLOv8m", "YOLOv8m-seg"]
engine = ModelInferenceEngine()
print("Loaded models:", {k: (v.mode if hasattr(v, "mode") else "yolo") for k, v in engine.models.items()})


## Vectorized per-image metric engine

In [ ]:
def to_binary(prob, thresh=0.5):
    return (prob >= thresh).astype(np.uint8)


def dice_score(pred_bin, gt_bin, eps=1e-6):
    inter = np.logical_and(pred_bin, gt_bin).sum()
    return float((2 * inter + eps) / (pred_bin.sum() + gt_bin.sum() + eps))


def iou_score(pred_bin, gt_bin, eps=1e-6):
    inter = np.logical_and(pred_bin, gt_bin).sum()
    union = np.logical_or(pred_bin, gt_bin).sum()
    return float((inter + eps) / (union + eps))


def mae_score(prob, gt_bin):
    return float(np.mean(np.abs(prob - gt_bin.astype(np.float32))))


def _boundary_band(mask, px=3):
    mask = mask.astype(np.uint8)
    kernel = np.ones((2 * px + 1, 2 * px + 1), np.uint8)
    dil = cv2.dilate(mask, kernel)
    ero = cv2.erode(mask, kernel)
    return dil - ero


def boundary_iou(pred_bin, gt_bin, px=3, eps=1e-6):
    pb, gb = _boundary_band(pred_bin, px), _boundary_band(gt_bin, px)
    inter = np.logical_and(pb, gb).sum()
    union = np.logical_or(pb, gb).sum()
    return float((inter + eps) / (union + eps))


def morphology_proxies(pred_bin):
    h, w = pred_bin.shape
    out = dict(abs_area=0.0, rel_area=0.0, solidity=0.0, circularity=0.0, aspect_ratio=0.0)
    contours, _ = cv2.findContours(pred_bin, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return out, None
    c = max(contours, key=cv2.contourArea)
    area = cv2.contourArea(c)
    perim = cv2.arcLength(c, True)
    hull = cv2.convexHull(c)
    hull_area = cv2.contourArea(hull)
    x, y, bw, bh = cv2.boundingRect(c)
    out["abs_area"] = float(area)
    out["rel_area"] = float(area / (h * w))
    out["solidity"] = float(area / hull_area) if hull_area > 0 else 0.0
    out["circularity"] = float(4 * np.pi * area / (perim ** 2)) if perim > 0 else 0.0
    out["aspect_ratio"] = float(bw / bh) if bh > 0 else 0.0
    return out, c


def illumination_texture(img_bgr, pred_bin):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    specular_ratio = float(np.mean(gray > 240))

    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB).astype(np.float32)
    kernel = np.ones((41, 41), np.uint8)
    ring = cv2.dilate(pred_bin, kernel) - pred_bin
    delta_e = 0.0
    if pred_bin.sum() > 0 and ring.sum() > 0:
        lesion_mean = lab[pred_bin.astype(bool)].mean(axis=0)
        ring_mean = lab[ring.astype(bool)].mean(axis=0)
        delta_e = float(np.linalg.norm(lesion_mean - ring_mean))
    return dict(specular_ratio=specular_ratio, mucosal_delta_e=delta_e)


def rdp_polygon(contour, epsilon_frac=0.01):
    if contour is None:
        return []
    perim = cv2.arcLength(contour, True)
    eps = max(epsilon_frac * perim, 1.0)
    approx = cv2.approxPolyDP(contour, eps, True)
    return approx.reshape(-1, 2).tolist()


def compute_all_metrics(prob, gt_bin, img_bgr, thresh=0.5):
    pred_bin = to_binary(prob, thresh)
    metrics = dict(
        dice=dice_score(pred_bin, gt_bin),
        iou=iou_score(pred_bin, gt_bin),
        mae=mae_score(prob, gt_bin),
        boundary_iou=boundary_iou(pred_bin, gt_bin),
        pred_fg_ratio=float(pred_bin.mean()),
    )
    morph, contour = morphology_proxies(pred_bin)
    metrics.update(morph)
    metrics.update(illumination_texture(img_bgr, pred_bin))
    metrics["polygon"] = rdp_polygon(contour)
    return metrics


## Benchmark loop: 5 test subsets + 200 random negative frames, across all 5 models

In [ ]:
import pandas as pd
import random


def load_gt_mask(mask_path):
    m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if m is None:
        return None
    return (m > 127).astype(np.uint8)


def get_images(folder):
    valid_exts = {".png", ".jpg", ".jpeg"}
    if not os.path.isdir(folder):
        return []
    return [
        os.path.join(folder, f) for f in os.listdir(folder)
        if os.path.splitext(f)[1].lower() in valid_exts
    ]

def iter_test_pairs(subset):
    img_dir = os.path.join(TEST_ROOT, subset, "images")
    mask_dir = os.path.join(TEST_ROOT, subset, "masks")
    
    img_paths = sorted(get_images(img_dir))
    for img_path in img_paths:
        stem = os.path.splitext(os.path.basename(img_path))[0]
        # Look for a mask with the same name across common extensions
        mask_candidates = [
            os.path.join(mask_dir, f"{stem}{ext}") 
            for ext in [".png", ".jpg", ".jpeg"]
        ]
        mask_path = next((p for p in mask_candidates if os.path.exists(p)), None)
        if mask_path:
            yield img_path, mask_path

# Update negative path discovery as well
random.seed(42)
neg_paths = sorted(get_images(NEG_ROOT))
neg_sample = random.sample(neg_paths, min(200, len(neg_paths))) if neg_paths else []
records = []


def run_one(model_name, img_path, mask_path, subset):
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        return
    prob = engine.predict(model_name, img_bgr)
    if mask_path is not None:
        gt_bin = load_gt_mask(mask_path)
        if gt_bin is None:
            return
        m = compute_all_metrics(prob, gt_bin, img_bgr)
        m.update(model=model_name, subset=subset, image=os.path.basename(img_path), is_negative=False)
    else:
        pred_bin = to_binary(prob)
        m = dict(
            model=model_name, subset="negatives_polypgen_v1",
            image=os.path.basename(img_path), is_negative=True,
            pred_fg_ratio=float(pred_bin.mean()),
            false_positive=bool(pred_bin.sum() > 0),
        )
    records.append(m)


for subset in SUBSETS:
    pairs = list(iter_test_pairs(subset))
    print(f"[{subset}] {len(pairs)} image/mask pairs")
    for img_path, mask_path in pairs:
        for model_name in MODEL_NAMES:
            run_one(model_name, img_path, mask_path, subset)

print(f"[negatives_polypgen_v1] {len(neg_sample)} sampled frames")
for img_path in neg_sample:
    for model_name in MODEL_NAMES:
        run_one(model_name, img_path, None, "negatives_polypgen_v1")

df = pd.DataFrame(records)
print("Total records:", df.shape)


## Export results and print the summary table

In [ ]:
out_parquet = "/kaggle/working/leakbench_per_image_metrics.parquet"
out_csv = "/kaggle/working/leakbench_per_image_metrics.csv"

df.to_parquet(out_parquet, index=False)
df.to_csv(out_csv, index=False)
print(f"Saved: {out_parquet}")
print(f"Saved: {out_csv}")

pos_df = df[df["is_negative"] == False]
neg_df = df[df["is_negative"] == True]

perf = pos_df.groupby("model")[["dice", "iou", "boundary_iou", "mae"]].mean()

if not neg_df.empty and "false_positive" in neg_df.columns:
    fpr = neg_df.groupby("model")["false_positive"].mean().rename("fpr_on_negatives")
    summary = perf.join(fpr).sort_values("dice", ascending=False).round(4)
else:
    print("Warning: No negative frames found; skipping false positive rate calculation.")
    summary = perf.sort_values("dice", ascending=False).round(4)

per_subset_dice = pos_df.pivot_table(index="model", columns="subset", values="dice", aggfunc="mean").round(4)

print("\n## LeakBench Model Comparison Summary\n")
print(summary.to_markdown())
print("\n## Per-subset Dice (seen-source: kvasir/clinicdb, unseen-source: colondb/cvc300/etis)\n")
print(per_subset_dice.to_markdown())